This routine builds the cached scenario ensemble used by every figure notebook in this repository, for the paper *"Global Potential of Potable Reuse Across Coupled Climate and Socioeconomic Futures"*. If you have any questions please contact [a.sarfraz@uu.nl](mailto:a.sarfraz@uu.nl).

## Preprocessing

The raw GCAM ensemble is stored as 459 scenario folders under `data/Scenarios/`, each holding several GCAM query parquets.

### Imports

In [1]:
import pandas as pd

### Setting directories

In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import yaml

# Resolve the repo root from this notebook's location.
_NB_DIR   = Path.cwd()
REPO_ROOT = _NB_DIR.parent if _NB_DIR.name == "notebooks" else _NB_DIR

sys.path.insert(0, str(REPO_ROOT / "src"))

# Load path config.
_cfg = yaml.safe_load((REPO_ROOT / "config" / "paths.yaml").read_text())

def _resolve(p):
    p = Path(p)
    return p if p.is_absolute() else (REPO_ROOT / p).resolve()

PATHS = {
    "data":    {k: _resolve(v) for k, v in _cfg["data"].items()},
    "outputs": {k: _resolve(v) for k, v in _cfg["outputs"].items()},
}

# Apply the manuscript-wide matplotlib style (axis labels 16 pt, no grid,
# 1200 dpi for raster output) and import the shared utilities.
from potable_reuse.style import apply_style
from potable_reuse.io import load_scenario_query
from potable_reuse.plotting import save_all
apply_style()

SCENARIOS_DIR = PATHS["data"]["scenarios_dir"]
CACHE_DIR     = PATHS["data"]["cache_dir"]
CACHE_DIR.mkdir(parents=True, exist_ok=True)

### Main code

Each call below walks every scenario folder, extracts the named GCAM query, and concatenates the result with the scenario design dimensions (SSP, RCP, reuse cost, supply capacity, PR level) appended as columns. The concatenated frame is written to `data/cache/`.

In [3]:
# Municipal water demand (used by Figure 1a, 1b boxplots).
muni = load_scenario_query(
    SCENARIOS_DIR,
    query_file="water_td_muni_W.parquet",
    cache_path=CACHE_DIR / "combined_muni.parquet",
)
print(f"muni  : {len(muni):,} rows | "
      f"{muni['scenario'].nunique()} scenarios")

  459 scenario folders matched, 459 contained the query, 323,136 total rows
  Cached: combined_muni.parquet
muni  : 323,136 rows | 459 scenarios


In [4]:
# Total freshwater withdrawals decomposed by sector and source
# (used by Figures 1a, 1b, 1c, 1d and the variance decomposition).
total = load_scenario_query(
    SCENARIOS_DIR,
    query_file="water withdrawals by water mapping source.parquet",
    cache_path=CACHE_DIR / "combined_total.parquet",
)
print(f"total : {len(total):,} rows | "
      f"{total['scenario'].nunique()} scenarios")

  459 scenario folders matched, 459 contained the query, 5,074,704 total rows
  Cached: combined_total.parquet
total : 5,074,704 rows | 459 scenarios
